# TCGA-BRCA Baseline Profile V1 Review

This notebook is review-only. It reads saved baseline-profile outputs from disk, regenerates review tables in `05-results`, and does not parse raw files, freeze the endpoint, add treatment detail, or perform modeling.


## Load the latest saved baseline-profile run and regenerate review tables


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'analysis-prep'
    / 'tcga_brca_baseline_profile_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest baseline-profile pointer not found: {latest_pointer_path}. Run the profile script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
field_summary_path = repo_root / latest_pointer['baseline_profile_v1_field_summary_tsv']
missingness_path = repo_root / latest_pointer['baseline_profile_v1_missingness_ranked_tsv']
value_summary_path = repo_root / latest_pointer['baseline_profile_v1_value_summary_tsv']
candidate_fields_path = repo_root / latest_pointer['baseline_profile_v1_candidate_fields_tsv']
excluded_fields_path = repo_root / latest_pointer['baseline_profile_v1_excluded_fields_tsv']
summary_path = repo_root / latest_pointer['baseline_profile_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

field_summary_df = read_tsv(field_summary_path)
missingness_df = read_tsv(missingness_path)
value_summary_df = read_tsv(value_summary_path)
candidate_fields_df = read_tsv(candidate_fields_path)
excluded_fields_df = read_tsv(excluded_fields_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError(f"Baseline-profile run is not completed: status={run_log.get('status')}")
if not run_log.get('validation', {}).get('passed', False):
    raise ValueError('Baseline-profile validation did not pass. Review the saved run_log.json before continuing.')

field_category_counts_df = (
    field_summary_df.groupby('field_category', dropna=False)
    .size()
    .reset_index(name='retained_field_count')
    .sort_values(['retained_field_count', 'field_category'], ascending=[False, True])
    .reset_index(drop=True)
)
candidate_bucket_counts_df = (
    candidate_fields_df.groupby('candidate_bucket', dropna=False)
    .size()
    .reset_index(name='field_count')
    .sort_values(['field_count', 'candidate_bucket'], ascending=[False, True])
    .reset_index(drop=True)
)

field_summary_review_df = field_summary_df.sort_values(
    ['field_category', 'source_origin', 'field_name']
).reset_index(drop=True)
missingness_review_df = (
    missingness_df.assign(
        missing_like_fraction_numeric=pd.to_numeric(missingness_df['missing_like_fraction'], errors='raise')
    )
    .sort_values(['missing_like_fraction_numeric', 'field_name'], ascending=[False, True])
    .drop(columns='missing_like_fraction_numeric')
    .reset_index(drop=True)
)
value_summary_review_df = value_summary_df.sort_values(['field_name']).reset_index(drop=True)
candidate_bucket_order = {
    'candidate_for_baseline_modeling_prep': 0,
    'candidate_but_review_needed': 1,
    'exclude_for_now': 2,
}
candidate_fields_review_df = (
    candidate_fields_df.assign(
        candidate_bucket_order=candidate_fields_df['candidate_bucket'].map(candidate_bucket_order).fillna(9),
        missing_like_fraction_numeric=pd.to_numeric(candidate_fields_df['missing_like_fraction'], errors='raise'),
        dominant_value_fraction_numeric=pd.to_numeric(candidate_fields_df['dominant_value_fraction'], errors='raise'),
    )
    .sort_values(
        ['candidate_bucket_order', 'field_category', 'field_name'],
        ascending=[True, True, True],
    )
    .drop(columns=['candidate_bucket_order', 'missing_like_fraction_numeric', 'dominant_value_fraction_numeric'])
    .reset_index(drop=True)
)
excluded_fields_review_df = excluded_fields_df.sort_values(
    ['excluded_reason', 'field_name']
).reset_index(drop=True)
summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)

strongest_baseline_candidates_df = candidate_fields_review_df.loc[
    candidate_fields_review_df['candidate_bucket'] == 'candidate_for_baseline_modeling_prep'
].reset_index(drop=True)
weak_or_unusable_df = candidate_fields_review_df.loc[
    candidate_fields_review_df['candidate_bucket'] != 'candidate_for_baseline_modeling_prep'
].reset_index(drop=True)
endpoint_and_biospecimen_review_df = candidate_fields_review_df.loc[
    candidate_fields_review_df['field_category'].isin([
        'endpoint_candidate_field',
        'followup_join_evidence_field',
        'biospecimen_sample_anchor_evidence_field',
    ])
].reset_index(drop=True)

field_summary_review_path = results_root / '80_baseline_profile_v1_field_summary.tsv'
missingness_review_path = results_root / '81_baseline_profile_v1_missingness_ranked.tsv'
value_summary_review_path = results_root / '82_baseline_profile_v1_value_summary.tsv'
candidate_fields_review_path = results_root / '83_baseline_profile_v1_candidate_fields.tsv'
excluded_fields_review_path = results_root / '84_baseline_profile_v1_excluded_fields.tsv'
summary_review_path = results_root / '85_baseline_profile_v1_summary.tsv'

field_summary_review_df.to_csv(field_summary_review_path, sep='\t', index=False)
missingness_review_df.to_csv(missingness_review_path, sep='\t', index=False)
value_summary_review_df.to_csv(value_summary_review_path, sep='\t', index=False)
candidate_fields_review_df.to_csv(candidate_fields_review_path, sep='\t', index=False)
excluded_fields_review_df.to_csv(excluded_fields_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f"Baseline profile run ID: {latest_pointer['baseline_profile_v1_run_id']}")
print(f"Baseline analysis run ID: {latest_pointer['baseline_analysis_v1_run_id']}")
print(f"Run log: {run_log_path}")
print(f"Saved: {field_summary_review_path}")
print(f"Saved: {missingness_review_path}")
print(f"Saved: {value_summary_review_path}")
print(f"Saved: {candidate_fields_review_path}")
print(f"Saved: {excluded_fields_review_path}")
print(f"Saved: {summary_review_path}")

display(pd.DataFrame([latest_pointer]))
display(pd.DataFrame([run_log.get('validation', {})]))
display(field_category_counts_df)
display(candidate_bucket_counts_df)
display(missingness_review_df.head(20))
display(strongest_baseline_candidates_df.head(20))
display(weak_or_unusable_df.head(20))
display(endpoint_and_biospecimen_review_df.head(20))
display(summary_review_df)


This notebook remains review-only. It must not be used to parse new raw inputs, perform imputation, collapse endpoint candidates into a final endpoint, reintroduce treatment detail, or train models.
